# Agentic AI Performance Evaluation Pipeline
**Richard Clay | A23CS0342 | SECP3843-01**

This notebook implements the data ingestion and transformation pipeline using Apache Spark (Medallion architecture).
Optimized to run on RAM-constrained VPS environments (driver memory = 1g, shuffle partitions = 4).

### --- Setup & Inisialisasi Spark Session

In [1]:
import os, time
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    current_timestamp, lit, col, lower, when, avg, count, 
    coalesce, expr, regexp_replace, round as spark_round
)

BASE_DIR = Path(r"C:\Users\richa\individual-project")
DATA_DIR = BASE_DIR / "data"
BRONZE_DIR = BASE_DIR / "03_Output_Bronze"
SILVER_DIR = BASE_DIR / "04_Output_Silver"
GOLD_DIR = BASE_DIR / "05_Output_Gold"

spark = SparkSession.builder \
    .appName("AgenticAI_Pipeline") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"Spark session initialized. Version: {spark.version}")

Spark session initialized. Version: 3.5.8


### --- 1. Bronze Layer (Ingestion)
Load raw data sources and save as Parquet.

In [2]:
bronze_start = time.time()

df_main = spark.read.csv(str(DATA_DIR / 'AgenticAI_Leadership_Dataset_v1.csv'), header=True, inferSchema=True)
df_bench = spark.read.option('multiLine', 'true').json(str(DATA_DIR / 'external_leadership_benchmarks.json'))
df_papers = spark.read.option('multiLine', 'true').json(str(DATA_DIR / 'openalex_ai_leadership_papers.json'))
df_logs = spark.read.csv(str(DATA_DIR / 'agent_execution_logs.csv'), header=True, inferSchema=True)

df_main_b = df_main.withColumn("ingestion_date", current_timestamp()).withColumn("data_source", lit("dataset_v1"))
df_bench_b = df_bench.withColumn("ingestion_date", current_timestamp()).withColumn("data_source", lit("external_benchmarks"))
df_papers_b = df_papers.withColumn("ingestion_date", current_timestamp()).withColumn("data_source", lit("openalex"))
df_logs_b = df_logs.withColumn("ingestion_date", current_timestamp()).withColumn("data_source", lit("execution_logs"))

df_main_b.write.mode("overwrite").parquet(str(BRONZE_DIR / "main_dataset"))
df_bench_b.write.mode("overwrite").parquet(str(BRONZE_DIR / "benchmark_data"))
df_papers_b.write.mode("overwrite").parquet(str(BRONZE_DIR / "openalex_papers"))
df_logs_b.write.mode("overwrite").parquet(str(BRONZE_DIR / "execution_logs"))

bronze_time = time.time() - bronze_start
print(f"Bronze: {bronze_time:.1f}s | Rows: {bronze_time:.1f}")

Bronze: 10.4s | Rows: 10.4


### --- 2. Silver Layer (Cleansing & NLP Mapping)
Median imputation per industry, benchmark key normalization, execution log merge, NLP citation mapping.

In [3]:
silver_start = time.time()

df_main_b = spark.read.parquet(str(BRONZE_DIR / 'main_dataset'))
df_bench_b = spark.read.parquet(str(BRONZE_DIR / 'benchmark_data'))
df_papers_b = spark.read.parquet(str(BRONZE_DIR / 'openalex_papers'))
df_logs_b = spark.read.parquet(str(BRONZE_DIR / 'execution_logs'))

df_main_b = df_main_b.withColumn('Industry', coalesce(col('Industry'), lit('unknown')))

# Median imputation per industry
median_map = {}
numeric_cols = ['Task_Success_Rate', 'Productivity_Improvement_Percent', 'Leadership_Trust_Score']
for c in numeric_cols:
    medians = df_main_b.groupBy('Industry').agg(expr(f'percentile_approx({c}, 0.5)').alias('med')).collect()
    median_map[c] = {row['Industry']: row['med'] for row in medians if row['med'] is not None}

for c in numeric_cols:
    mapping = median_map[c]
    expr_str = "when(col('" + c + "').isNull(), when(col('Industry') == '" + list(mapping.keys())[0] + "', " + str(mapping[list(mapping.keys())[0]]) + ")"
    for ind, val in list(mapping.items())[1:]:
        expr_str += f".when(col('Industry') == '{ind}', {val})"
    expr_str += f".otherwise(0)).otherwise(col('{c}'))"
    df_main_b = df_main_b.withColumn(c, eval(expr_str))

# Benchmark key normalization (underscore to space)
df_bench_clean = df_bench_b.withColumn('Industry_Key', regexp_replace(col('Industry'), '_', ' '))

df_joined = df_main_b.join(
    df_bench_clean.select('Industry_Key', 'Benchmark_Task_Success_Rate', 'Benchmark_Productivity_Improvement', 'Benchmark_Trust_Score'),
    df_main_b.Industry == col('Industry_Key'), how='left'
)

# Merge execution logs
df_logs_clean = df_logs_b.select('Record_ID', 'Execution_Time_Minutes', 'Error_Count')
df_joined = df_joined.join(df_logs_clean, on='Record_ID', how='left')

# NLP sector mapping
df_papers_mapped = df_papers_b.withColumn(
    'Mapped_Sector',
    when(lower(col('title')).rlike('finance|market|bank|stock|invest'), 'financial services')
    .when(lower(col('title')).rlike('health|medic|clinical|patient|hospital'), 'healthcare')
    .when(lower(col('title')).rlike('learn|school|teach|university|education'), 'education')
    .when(lower(col('title')).rlike('govern|policy|public|legal|state'), 'government')
    .when(lower(col('title')).rlike('tech|software|algorithm|compute|ai|data'), 'technology')
    .otherwise('other')
)
sector_citations = df_papers_mapped.groupBy('Mapped_Sector').agg(avg('citations').alias('Avg_Citations'))
df_final_silver = df_joined.join(sector_citations, df_joined.Industry == lower(sector_citations.Mapped_Sector), how='left')

df_final_silver.write.mode("overwrite").parquet(str(SILVER_DIR / "agentic_leadership_silver"))
silver_time = time.time() - silver_start
print(f"Silver: {silver_time:.1f}s | Rows: {df_final_silver.count()}")

Silver: 4.0s | Rows: 5500


### --- 3. Gold Layer (Star Schema Marts)
Derived metrics (Trust Gap, Productivity Category), dimension + fact table export.

In [4]:
gold_start = time.time()

df_silver = spark.read.parquet(str(SILVER_DIR / 'agentic_leadership_silver'))

df_silver_enriched = df_silver.withColumn(
    'Trust_vs_Benchmark_Gap',
    col('Leadership_Trust_Score') - col('Benchmark_Trust_Score')
).withColumn(
    'Productivity_Category',
    when(col('Productivity_Improvement_Percent') >= 18.0, 'High')
    .when(col('Productivity_Improvement_Percent') >= 10.0, 'Medium')
    .otherwise('Low')
)

dim_industry = df_silver_enriched.groupBy('Industry').agg(
    count('Record_ID').alias('Deployments'),
    spark_round(avg('Task_Success_Rate'), 2).alias('Avg_Success_Rate'),
    spark_round(avg('Productivity_Improvement_Percent'), 2).alias('Avg_Productivity'),
    spark_round(avg('Avg_Citations'), 2).alias('Research_Citation_Avg'),
    spark_round(avg('Trust_vs_Benchmark_Gap'), 2).alias('Trust_Gap')
)

fact_table = df_silver_enriched
dim_industry.write.mode("overwrite").parquet(str(GOLD_DIR / "industry_ai_summary"))
fact_table.write.mode("overwrite").parquet(str(GOLD_DIR / "ai_enriched_agentic_leadership"))

null_count = fact_table.filter(col("Benchmark_Trust_Score").isNull()).count()
assert null_count == 0, f"Null benchmark join: {null_count}"

gold_time = time.time() - gold_start
print(f"Gold: {gold_time:.1f}s | Dim: {dim_industry.count()} | Fact: {fact_table.count()}")

Gold: 1.4s | Dim: 11 | Fact: 5500


### --- 4. Summary & Shutdown

In [5]:
total = time.time() - bronze_start
print(f"Pipeline done: {total:.2f}s")
spark.stop()

Pipeline done: 17.19s
